In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_dialer_extfile(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

       # schema = (
        # "DialerId	string,"
        # "SourceSystem	string,"
        # "PortfolioKey	int,"
        # "PortfolioName	string,"
        # "IngestionTimestamp	timestamp,"
        # "BatchId	string,"
        # "DialerKey	string,"
        # "DialerCode	string,"
        # "ExposureKey	string,"
        # "PartyKey	string,"
        # "CDI	string,"
        # "CaseCode	string,"
        # "DialerType	string,"
        # "CampaignName	string,"
        # "CampaignTypeCode	string,"
        # "CampaignType	string,"
        # "LineTypeCode	string,"
        # "LineType	string,"
        # "SetupSourceCode	string,"
        # "IsActiveFlag	boolean,"
        # "IsExternalFlag	boolean,"
        # "ExternalCompanyCode	string,"
        # "ExternalCompany	string,"
        # "CallTypeCode	string,"
        # "CallType	string,"
        # "SIPResponseTypeCode	string,"
        # "SIPResponseType	string,"
        # "AgentPartyKey	string,"
        # "DateHour	timestamp,"
        # "CallDisposition	string,"
        # "RingTimeDuration	string,"
        # "TalkTimeDuration	string,"
        # "HungUpSite	string,"
        # "AnsweredTime	string,"
        # "WrapupTimeSec	string,"
        # "CallerPhone	string,"
        # "NumberCalled	    string,"
        # "TotalIvrTime	    int,"
        # "QueueEntered	    int,"
        # "SmartIvrFlowId	int,"
        # "CreatedAt	    timestamp,"
        # "IvrStageCode	    string,"
        # "Info        	    string,"
        # "Description	    string,"
        # "Message	        string,"
        # "Action	        string,"
        # "IvrMenu	        int,"
        # "NameFile	string,"
        # "NameFileAgent	string,"
        # "LastUpdatedDate	timestamp,"
        # "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Dialer"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "extfile",
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== dialerexternal ==================

        dialerexternal_extra_filters = None

        if dialerexternal_extra_filters is not None:
            dialerexternal_extra_filters.append(date_filter)
            dialerexternal_final_filters = " AND ".join(dialerexternal_extra_filters)
        else:
            dialerexternal_final_filters = date_filter

        dialerexternal_selected_fields = [
            "id",
            "cdi",
            "campaignname",
            "agentname",
            "datehour",
            "calldisposition",
            "ringtimeduration",
            "talktimeduration",
            "hungupsite",
            "answeredtime",
            "wrapuptime",
            "callerphone",
            "name_file",
            "name_file_agent",
            "lastupdatedate",
            "businessdate",
        ]

        dialerexternal = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "extfile_dialerexternal",
            dialerexternal_final_filters,
            dialerexternal_selected_fields,
        )

        # ================== dialersetup ==================

        dialersetup_extra_filters = None

        if dialersetup_extra_filters is not None:
            dialersetup_extra_filters.append(date_filter)
            dialersetup_final_filters = " AND ".join(dialersetup_extra_filters)
        else:
            dialersetup_final_filters = date_filter

        dialersetup_selected_fields = [
            "portfolioid",
            "campaign_type",
            "line_type",
            "source",
            "isactive",
            "is_external",
            "external_office_id",
            "campaign_name",
            "businessdate",
        ]

        dialersetup = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "params_dialersetup",
            dialersetup_final_filters,
            dialersetup_selected_fields,
        )

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "prefix",
            "portfolioname",
            "unit_code_profits",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # Join conditions
        dialerexternal_dialersetup_join_condition = (
            dialerexternal["campaignname"] == dialersetup["campaign_name"]
        ) & (dialersetup["campaign_type"] == "IN")

        dialerexternal_portfolio_join_condition = (
            substring(dialerexternal["CDI"], 1, 3) == portfolio["prefix"]
        )

        # Joins
        target_table_extfile = dialerexternal.join(
            dialersetup, dialerexternal_dialersetup_join_condition, "left"
        ).join(portfolio, dialerexternal_portfolio_join_condition, "left")

        # HashKeys
        target_table_extfile_DialerId_primary_key_fields = [
            dialerexternal.cdi,
            dialerexternal.campaignname,
            dialerexternal.datehour,
            dialerexternal.callerphone,
            dialerexternal.name_file,
            lit("extfile"),
        ]

        # Target table(hash keys and calculations)
        target_table_extfile = (
            target_table_extfile.withColumn(
                "DialerId",
                sha2(
                    concat_ws(";", *target_table_extfile_DialerId_primary_key_fields),
                    256,
                ),
            )
            .withColumn(
                "DialerKey",
                concat_ws(
                    ";",
                    dialerexternal.cdi,
                    dialerexternal.campaignname,
                    dialerexternal.datehour,
                    dialerexternal.callerphone,
                    dialerexternal.name_file,
                ),
            )
            .withColumn(
                "PortfolioKey",
                coalesce(portfolio["unit_code_profits"], lit(-9999)),
            )
            .withColumn(
                "PortfolioName",
                coalesce(portfolio["PortfolioName"], lit("Core Retail")),
            )
        )

        # ================== Selection of Final Columns ==================

        result_df_extfile = target_table_extfile.select(
            target_table_extfile.DialerId.alias("DialerId"),
            lit("extfile").cast("String").alias("SourceSystem"),
            target_table_extfile.PortfolioKey.alias("PortfolioKey"),
            target_table_extfile.PortfolioName.cast("String").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            target_table_extfile.DialerKey.cast("String").alias("DialerKey"),
            target_table_extfile.DialerKey.cast("String").alias("DialerCode"),
            lit(None).cast("String").alias("ExposureKey"),
            lit(None).cast("String").alias("PartyKey"),
            dialerexternal.cdi.cast("string").alias("CDI"),
            lit(None).cast("String").alias("CaseCode"),
            lit("extfile").alias("DialerType"),
            dialerexternal.campaignname.cast("string").alias("CampaignName"),
            dialersetup.campaign_type.cast("string").alias("CampaignTypeCode"),
            dialersetup.campaign_type.cast("String").alias("CampaignType"),
            dialersetup.line_type.cast("String").alias("LineTypeCode"),
            dialersetup.line_type.cast("String").alias("LineType"),
            dialersetup.source.cast("String").alias("SetupSourceCode"),
            dialersetup.isactive.cast("boolean").alias("IsActiveFlag"),
            dialersetup.is_external.cast("boolean").alias("IsExternalFlag"),
            dialersetup.external_office_id.cast("String").alias("ExternalCompanyCode"),
            dialersetup.external_office_id.cast("String").alias("ExternalCompany"),
            lit(None).cast("String").alias("CallTypeCode"),
            lit(None).cast("String").alias("CallType"),
            lit(None).cast("String").alias("SIPResponseTypeCode"),
            lit(None).cast("String").alias("SIPResponseType"),
            dialerexternal.agentname.cast("String").alias("AgentPartyKey"),
            dialerexternal.datehour.cast("timestamp").alias("DateHour"),
            dialerexternal.calldisposition.cast("String").alias("CallDisposition"),
            dialerexternal.ringtimeduration.cast("String").alias("RingTimeDuration"),
            dialerexternal.talktimeduration.cast("String").alias("TalkTimeDuration"),
            dialerexternal.hungupsite.cast("String").alias("HungUpSite"),
            dialerexternal.answeredtime.cast("String").alias("AnsweredTime"),
            dialerexternal.wrapuptime.cast("String").alias("WrapupTimeSec"),
            dialerexternal.callerphone.cast("String").alias("CallerPhone"),
            lit(None).alias("NumberCalled"),
            lit(None).alias("TotalIvrTime"),
            lit(None).alias("QueueEntered"),
            dialerexternal.name_file.cast("String").alias("NameFile"),
            dialerexternal.name_file_agent.cast("String").alias("NameFileAgent"),
            dialerexternal.lastupdatedate.cast("timestamp").alias("LastUpdatedDate"),
            dialerexternal.businessdate.cast("Date").alias("BusinessDate"),
        )

        # ================== Append ==================

        result_df_extfile.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_dialer_extfile(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)